# 🏭 VELARA Design Team — Kaggle GPU Worker

**Setup (ek baar, 3 minute):**
1. Ye notebook Kaggle mein import karo — is URL ko kholo:
   `https://www.kaggle.com/kernels/welcome?src=https://github.com/priyanshagrahari54-blip/Velara/blob/main/kaggle/VELARA-Designer.ipynb`
2. Right panel **Settings ▸ Accelerator ▸ GPU T4 x2** (ya P100) select karo.
3. **Add-ons ▸ Secrets ▸ Add a secret**: Label = `GHTOKEN`, Value = aapka GitHub token
   (GitHub ▸ Settings ▸ Developer settings ▸ Fine-grained tokens ▸ sirf **Velara** repo ▸ Contents: Read+Write).
   *Secret nahi diya to job chalega, lekin push nahi — assistant tmate se kheenchega.*
4. **Run All** — worker queue se job uthake GPU pe design karega aur khud push karega.

**Kaggle limits:** ~12h/session, ~30 GPU-ghante/hafte. Tab band na karo jab tak "RESULT" na print ho.


In [ ]:
# platform: Kaggle
WORKER_NAME = 'kaggle-gpu'
import os
os.environ['VELARA_WORK'] = '/kaggle/working'
def GET_TOKEN():
    from kaggle_secrets import UserSecretsClient
    try:
        return UserSecretsClient().get_secret('GHTOKEN') or ''
    except Exception:
        return ''

# ================================================================
#  VELARA Design Team — GPU worker (queue-aware)
#  Run All dabao. Ye worker factory/queue se pending job uthata
#  hai, Blender GPU pe chalata hai, verify karta hai — aur agar
#  GHTOKEN secret set hai to result khud repo mein push karta hai.
# ================================================================
import subprocess, os, time, json, glob, sys

def run(cmd, cwd=None):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if r.stdout.strip(): print(r.stdout.strip()[:600])
    if r.returncode != 0 and r.stderr.strip(): print('ERR:', r.stderr.strip()[:400])
    return r.returncode

WORK = os.environ.get('VELARA_WORK', os.path.abspath('.'))
print('== 1) GPU check ==')
g = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv',
                   shell=True, capture_output=True, text=True)
print(g.stdout.strip() if g.returncode == 0 else 'NO GPU — CPU pe chalega (slow)')

print('\n== 2) deps + Blender 4.5.16 ==')
run('apt-get -qq update')
run('apt-get -qq install -y libx11-6 libxi6 libxxf86vm1 libxfixes3 libxrender1 '
    'libgl1 libxkbcommon0 libsm6 libice6 xz-utils git')
if not os.path.exists('/opt/blender/blender'):
    run('curl -fsSL -o /tmp/b.tar.xz '
        'https://download.blender.org/release/Blender4.5/blender-4.5.16-linux-x64.tar.xz')
    run('mkdir -p /opt/blender && tar -xJf /tmp/b.tar.xz -C /opt/blender --strip-components=1 && rm /tmp/b.tar.xz')
v = subprocess.run('/opt/blender/blender --version', shell=True, capture_output=True, text=True)
print(v.stdout.splitlines()[0] if v.returncode == 0 and v.stdout else 'BLENDER FAIL')

print('\n== 3) token (optional secret) ==')
TOKEN = ''
try:
    TOKEN = GET_TOKEN()  # platform-specific, upar define kiya gaya hai
    print('token mila — autonomous push ON' if TOKEN else 'token nahi — assistant pull karega')
except Exception as e:
    print('secret nahi mila:', str(e)[:120])

print('\n== 4) repo clone/pull ==')
REPO_URL = 'https://github.com/priyanshagrahari54-blip/Velara.git'
repo = os.path.join(WORK, 'Velara')
if not os.path.exists(repo):
    clone = REPO_URL if not TOKEN else REPO_URL.replace('https://', f'https://x-access-token:{TOKEN}@')
    run(f'git clone {clone} {repo}')
run('git pull --ff-only -q', cwd=repo)

print('\n== 5) claim next pending job ==')
qfiles = sorted(glob.glob(os.path.join(repo, 'factory', 'queue', '*.json')))
jobs = [(f, json.load(open(f))) for f in qfiles]
now = time.time()
pend = [(f, j) for f, j in jobs
        if j.get('status') == 'pending'
        or (j.get('status') == 'running' and now - j.get('claimed', 0) > 3600)]
pend.sort(key=lambda fj: (fj[1].get('priority', 5), fj[1]['id']))
if not pend:
    print('QUEUE KHALI — koi pending job nahi. DM (assistant) se naye jobs maango.')
else:
    f, job = pend[0]
    job['status'] = 'running'; job['worker'] = WORKER_NAME; job['claimed'] = int(now)
    json.dump(job, open(f, 'w'), indent=1)
    print(f"JOB: {job['id']} — {job.get('brief','')[:100]}")

    print('\n== 6) Blender GPU job (heavy-quality bake) ==')
    args = ' '.join(job.get('args', []))
    rc = run(f'/opt/blender/blender --background --python factory/jobs/{job["script"]} '
             f'-- {args} bake 2048 ao 1024 samples 256', cwd=repo)
    glb = os.path.join(repo, 'factory', 'out', job['id'] + '.glb')
    if rc == 0 and os.path.exists(glb):
        vrc = run(f'/opt/blender/blender --background --python factory/verify-glb.py -- {glb}', cwd=repo)
        ok = vrc == 0
    else:
        ok = False
    job['status'] = 'done' if ok else 'failed'
    if ok:
        job['result'] = f"factory/out/{job['id']}.glb"
        job['bytes'] = os.path.getsize(glb)
        job['worker'] = WORKER_NAME
    json.dump(job, open(f, 'w'), indent=1)
    print(f"RESULT: {job['id']} -> {job['status']}")

    if ok and TOKEN:
        print('\n== 7) push results to repo ==')
        run('git config user.email designer@worker && git config user.name designer-gpu', cwd=repo)
        run('git add factory/out factory/queue', cwd=repo)
        run('git commit -q -m "designer(gpu): ' + job['id'] + ' result [skip ci]" || true', cwd=repo)
        run('git pull --rebase -q origin main || true', cwd=repo)
        push = REPO_URL.replace('https://', f'https://x-access-token:{TOKEN}@')
        run(f'git push {push} HEAD:main', cwd=repo)
    elif ok:
        print('\nResult file:', glb, '— assistant ko bolo, tmate/download se kheenchega.')
